# 🎓 Técnicas Robustas

En este tema aprenderemos técnicas avanzadas de segmentación. La segmentación es el proceso de separar el objeto de interés del fondo. Hasta ahora hemos usado color, pero ¿qué pasa si el color no es suficiente o la luz cambia?

### 🎯 Objetivos de Aprendizaje
1.  Comprender la **Umbralización (Thresholding)** simple.
2.  Usar el método de **Otsu** para encontrar umbrales automáticamente.
3.  Aplicar **Umbralización Adaptativa** para documentos con mala iluminación.

---

## 1. Umbralización Simple (Thresholding)

Es la forma más básica de segmentación. Si un píxel es más brillante que un valor $T$, lo hacemos blanco (255); si no, negro (0).

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Cargar imagen en escala de grises (ej. un degradado o un objeto claro sobre fondo oscuro)
imagen = cv2.imread('gradiente.jpg', 0) 
if imagen is None:
    # Crear degradado sintético 0-255
    imagen = np.tile(np.arange(256, dtype=np.uint8), (200,1))

# Aplicar Threshold (Umbral = 127)
# Tipos: THRESH_BINARY, THRESH_BINARY_INV, THRESH_TRUNC, etc.
ret, thresh1 = cv2.threshold(imagen, 127, 255, cv2.THRESH_BINARY)
ret, thresh2 = cv2.threshold(imagen, 127, 255, cv2.THRESH_BINARY_INV)

plt.figure(figsize=(10, 5))
plt.subplot(1, 3, 1); plt.imshow(imagen, cmap='gray'); plt.title("Original")
plt.subplot(1, 3, 2); plt.imshow(thresh1, cmap='gray'); plt.title("Binary (T>127)")
plt.subplot(1, 3, 3); plt.imshow(thresh2, cmap='gray'); plt.title("Binary Inv")
plt.show()

## 2. Binarización de Otsu

¿Cómo elegimos el número mágico 127? **Otsu** es un algoritmo que analiza el histograma de la imagen y encuentra automáticamente el umbral óptimo que separa dos picos (fondo y objeto).

In [ ]:
# Imagen ruidosa bimodal (ej. texto escaneado)
img_ruido = cv2.imread('texto_ruido.jpg', 0)
if img_ruido is None:
    # Crear imagen sintética bimodal con ruido
    img_ruido = np.zeros((300, 300), dtype=np.uint8)
    img_ruido[:, :150] = 50  # Fondo oscuro
    img_ruido[:, 150:] = 200 # Objeto claro
    cv2.randn(img_ruido, 127, 20) # Añadir ruido

# 1. Threshold Global (Valor arbitrario 127)
ret1, th1 = cv2.threshold(img_ruido, 127, 255, cv2.THRESH_BINARY)

# 2. Threshold de Otsu (Pasamos 0 como umbral y el flag THRESH_OTSU)
ret2, th2 = cv2.threshold(img_ruido, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

print(f"Umbral calculado por Otsu: {ret2}")

plt.figure(figsize=(10, 8))
plt.subplot(2, 2, 1); plt.imshow(img_ruido, cmap='gray'); plt.title("Original Ruidosa")
plt.subplot(2, 2, 2); plt.hist(img_ruido.ravel(), 256); plt.title("Histograma")
plt.subplot(2, 2, 3); plt.imshow(th1, cmap='gray'); plt.title("Global (127)")
plt.subplot(2, 2, 4); plt.imshow(th2, cmap='gray'); plt.title("Otsu (Automático)")
plt.show()

## 3. Umbralización Adaptativa

**Problema:** Si la imagen tiene sombras (iluminación no uniforme), un umbral global (como Otsu) fallará. Parte de la sombra se detectará como objeto.

**Solución:** `cv2.adaptiveThreshold` calcula el umbral para cada pequeña región de la imagen.

In [ ]:
# Cargar imagen de Sudoku (clásico ejemplo con sombras)
img_sudoku = cv2.imread('sudoku.jpg', 0)
if img_sudoku is None:
    img_sudoku = np.tile(np.arange(256, dtype=np.uint8), (256,1)) # Fallback

# 1. Threshold Global (Falla en las sombras)
_, th_global = cv2.threshold(img_sudoku, 127, 255, cv2.THRESH_BINARY)

# 2. Adaptativo Gaussiano
# Argumentos: (src, maxVal, metodo_adaptativo, tipo_thresh, tamaño_bloque, constante_C)
th_adapt = cv2.adaptiveThreshold(img_sudoku, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, \
                                 cv2.THRESH_BINARY, 11, 2)

plt.figure(figsize=(12, 5))
plt.subplot(1, 3, 1); plt.imshow(img_sudoku, cmap='gray'); plt.title("Original con Sombras")
plt.subplot(1, 3, 2); plt.imshow(th_global, cmap='gray'); plt.title("Global (Falla)")
plt.subplot(1, 3, 3); plt.imshow(th_adapt, cmap='gray'); plt.title("Adaptativo (Éxito)")
plt.show()

## 4. 🚀 Mini-Proyecto: Scanner de Documentos

**Objetivo:** Simula una app de escaneo (como CamScanner) para procesar una foto de un documento mal iluminado.

**Pasos:**
1.  Convertir a escala de grises.
2.  Aplicar un poco de Blur (para reducir el ruido del papel).
3.  Aplicar **Adaptive Threshold** para binarizar el texto limpiamente, ignorando sombras.
4.  (Opcional) Usar operaciones morfológicas para limpiar puntitos negros.

In [1]:
# TODO: Implementa tu scanner aquí

import cv2
import numpy as np

# --- 1. Generación de Imagen de Prueba (Documento con Sombra) ---
# Creamos una imagen que simula una hoja de papel mal iluminada
# Generamos un gradiente de sombra (de oscuro a claro)
ancho, alto = 600, 800
x = np.linspace(0, 1, ancho)
y = np.linspace(0, 1, alto)
X, Y = np.meshgrid(x, y)
sombra = (255 * (0.4 + 0.6 * X * Y)).astype(np.uint8) # Gradiente diagonal
imagen = cv2.merge([sombra, sombra, sombra])

# Añadimos texto negro (simulando tinta)
fuente = cv2.FONT_HERSHEY_SIMPLEX
cv2.putText(imagen, "FACTURA 001", (50, 100), fuente, 2, (0, 0, 0), 4)
cv2.putText(imagen, "Cliente: Juan Perez", (50, 200), fuente, 1, (20, 20, 20), 2)
cv2.putText(imagen, "Total: $500.00", (50, 300), fuente, 1.5, (0, 0, 0), 3)
cv2.putText(imagen, "Firma: ___________", (50, 600), fuente, 1, (10, 10, 10), 2)

# Añadir un poco de ruido de papel
ruido = np.random.normal(0, 5, imagen.shape).astype(np.uint8)
imagen = cv2.add(imagen, ruido)

print("🖨️ Procesando documento con sombras...")

# --- 2. Lógica del Scanner ---

def escanear_documento(img):
    # Paso 1: Convertir a escala de grises
    # Necesario porque threshold solo funciona en 1 canal
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Paso 2: Reducir ruido del papel (Blur)
    # Usamos MedianBlur porque es excelente para quitar el "grano" del papel
    # sin borrar los bordes definidos de las letras.
    gris_suave = cv2.medianBlur(gris, 5)
    
    # Paso 3: Binarización con Umbral Adaptativo (La magia)
    # En lugar de un valor fijo (ej. 127), el algoritmo calcula el umbral
    # para cada vecindario de 11x11 píxeles.
    # C=2 es una constante que se resta al promedio para asegurar que el fondo
    # (que es el promedio) se vuelva blanco y el texto (más oscuro) negro.
    escaneo = cv2.adaptiveThreshold(
        gris_suave,          # Imagen de entrada
        255,                 # Valor máximo (blanco)
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C, # Método (Gaussiano es más natural)
        cv2.THRESH_BINARY,   # Tipo de umbral
        11,                  # Tamaño del bloque (vecindario)
        2                    # Constante C
    )
    
    # Paso 4 (Opcional): Limpieza morfológica
    # A veces quedan puntitos negros pequeños (ruido). 
    # Una "Apertura" (Erosión seguida de Dilatación) los elimina.
    # Nota: En binario (letras negras sobre blanco), la lógica se invierte
    # o simplemente aplicamos sobre el resultado.
    
    return escaneo

# --- 3. Ejecución y Visualización ---

resultado = escanear_documento(imagen)

# Para visualizar mejor, apilamos las imágenes
# Convertimos el resultado (gris) a BGR para poder apilarlo con la original
resultado_bgr = cv2.cvtColor(resultado, cv2.COLOR_GRAY2BGR)
comparativa = np.hstack((imagen, resultado_bgr))

cv2.imshow("Izquierda: Original con Sombra | Derecha: Escaneo Limpio", comparativa)

print("✅ Documento escaneado exitosamente.")
print("Nota: Observa cómo la sombra desaparece completamente en la versión derecha.")
print("Presiona cualquier tecla para cerrar...")

cv2.waitKey(0)
cv2.destroyAllWindows()

🖨️ Procesando documento con sombras...
✅ Documento escaneado exitosamente.
Nota: Observa cómo la sombra desaparece completamente en la versión derecha.
Presiona cualquier tecla para cerrar...
